In [1]:
import argparse
from pathlib import Path
import math


In [20]:
class createFiles:
    def __init__(self, m: int):
        self.m = m
        self.entity_name = f"sum{m}xn"

    def is_power_of_two(self, x: int) -> bool:
        return x >= 2 and (x & (x - 1)) == 0


    def chunks(self, lista, tamanho):
        for i in range(0, len(lista), tamanho):
            yield lista[i:i + tamanho]


    def format_port_list(self, ports, indent="        ", per_line=4):
        lines = []

        for group in self.chunks(ports, per_line):
            lines.append(indent + ", ".join(group))

        return ",\n".join(lines)
    
    
    def format_port_decl(self, names, indent="        ", per_line=4):
        lines = []
        port_w = max(len(p) for p in names)

        for i in range(0, len(names), per_line):
            group = names[i:i + per_line]
            line = ", ".join(f"{p:<{port_w}}" for p in group)

            if i + per_line < len(names):
                line += ","

            lines.append(f"{indent}{line}")

        return "\n".join(lines)


    def format_a_in_aggregate(self, ports):
        """
        Monta o aggregate VHDL que transforma a0, a1, ... em a_in(0), a_in(1), ...
        """
        lines = []

        for i in range(0, len(ports), 4):
            group = []

            for j in range(i, min(i + 4, len(ports))):
                comma = "," if j < len(ports) - 1 else ""
                group.append(f"{j:3d} => {ports[j]}{comma}")

            lines.append("        " + " ".join(group))

        return "\n".join(lines)


    def format_port_map_msb(self, ports):
        """
        Mapeia a tree_msb usando as entradas ja deslocadas no TOP.
        Antes era:
            a0 => a0(n-1 downto n/2-h)
        Agora fica:
            a0 => a_shift(0)(n-1 downto n/2-h)
        """
        lines = []
        port_w = max(len(p) for p in ports)

        for i, p in enumerate(ports):
            lines.append(
                f"            {p:<{port_w}} => a_shift({i})(n-1 downto n/2-h),"
            )

        return "\n".join(lines)


    def format_port_map_lsb(self, ports):
        """
        Mapeia a tree_lsb usando as entradas ja deslocadas no TOP.
        Antes era:
            a0 => a0(n/2-h-1 downto 0)
        Agora fica:
            a0 => a_shift(0)(n/2-h-1 downto 0)
        """
        lines = []
        port_w = max(len(p) for p in ports)

        for i, p in enumerate(ports):
            lines.append(
                f"            {p:<{port_w}} => a_shift({i})(n/2-h-1 downto 0),"
            )

        return "\n".join(lines)
    
    def generate_sum_vhdl(self, m: int) -> str:
        if not self.is_power_of_two(m):
            raise ValueError("O número de vetores M deve ser potência de 2: 2, 4, 8, 16, 32...")

        if m < 4:
            raise ValueError("Esta versão particionada MSB/LSB faz mais sentido para M >= 4.")

        entity_name = f"sum{m}xn"

        # Portas internas das árvores continuam sendo a0, a1, ..., aM-1
        ports = [f"a{i}" for i in range(m)]

        h = int(math.log2(m))
        last_level = h - 1

        f_msb_0 = f"f{last_level}_0"
        f_msb_1 = f"f{last_level}_1"

        port_decl_comp = self.format_port_decl(ports, indent="            ")

        # Mapeia as entradas já shiftadas para as árvores originais
        port_map_msb = []
        port_map_lsb = []
        port_w = max(len(p) for p in ports)

        for i, p in enumerate(ports):
            port_map_msb.append(
                f"            {p:<{port_w}} => a_shift({i})(n-1 downto n/2-h),"
            )
            port_map_lsb.append(
                f"            {p:<{port_w}} => a_shift({i})(n/2-h-1 downto 0),"
            )

        port_map_msb = "\n".join(port_map_msb)
        port_map_lsb = "\n".join(port_map_lsb)

        vhdl = f"""
library ieee;
use ieee.std_logic_1164.all;
use ieee.numeric_std.all;

entity {entity_name} is
    generic(
        n : integer := {m};
        h : integer := {h}
    );
    port (
        a0  : in  unsigned(n-1 downto 0);
        a1  : in  unsigned(n-1 downto 0);
        sum : out unsigned(n-1 downto 0)
    );
end {entity_name};

architecture rtl of {entity_name} is

    constant M : integer := {m};

    type input_array_t is array (0 to M-1) of unsigned(n-1 downto 0);

    signal pp      : input_array_t;
    signal a_shift : input_array_t;

    component tree_msb is
        generic(n : integer := 16);
        port (
{port_decl_comp} : in unsigned(n-1 downto 0);
            {f_msb_1}, {f_msb_0} : out unsigned(n-1 downto 0);
            sum : out unsigned(n-1 downto 0)
        );
    end component;

    component tree_lsb is
        generic(n : integer := 16);
        port (
{port_decl_comp} : in unsigned(n-1 downto 0);
            sum : out unsigned(n+{h-1} downto 0)
        );
    end component;

    --------------------------------------------------------------------------
    -- Saídas da árvore MSB
    --------------------------------------------------------------------------
    signal {f_msb_0}_msb, {f_msb_1}_msb : unsigned(n/2+h-1 downto 0);
    signal sum_tree_msb : unsigned(n/2+h-1 downto 0);

    --------------------------------------------------------------------------
    -- Saída da árvore LSB
    --------------------------------------------------------------------------
    signal sum_tree_lsb : unsigned(n/2-1 downto 0);

    --------------------------------------------------------------------------
    -- Carry parcial entre LSB e MSB
    --------------------------------------------------------------------------
    signal csum : unsigned(h downto 0);

    --------------------------------------------------------------------------
    -- Sinais do CSLA
    --------------------------------------------------------------------------
    signal sum_msb : unsigned(n/2-1 downto 0);

    signal sum_c0 : unsigned(n/2-1 downto 0);
    signal sum_c1 : unsigned(n/2 downto 0);

begin

    --------------------------------------------------------------------------
    -- Produtos parciais truncados em n bits
    --
    -- pp(i) = a0 se a1(i) = '1'
    -- pp(i) = 0  se a1(i) = '0'
    --------------------------------------------------------------------------
    gen_pp : for i in 0 to M-1 generate
        pp(i) <= a0 when a1(i) = '1' else (others => '0');
    end generate;

    --------------------------------------------------------------------------
    -- Deslocamento diagonal feito no TOP
    --
    -- a_shift(0)   = pp(0)
    -- a_shift(1)   = pp(1) sll 1
    -- ...
    -- a_shift(M-1) = pp(M-1) sll M-1
    --
    -- Como a_shift tem n bits, o resultado final fica truncado em n bits.
    --------------------------------------------------------------------------
    gen_shift : for i in 0 to M-1 generate
        a_shift(i) <= shift_left(pp(i), i);
    end generate;

    --------------------------------------------------------------------------
    -- Árvore MSB original
    --------------------------------------------------------------------------
    u_tree_msb : tree_msb
        generic map(
            n => n/2 + h
        )
        port map(
{port_map_msb}
            {f_msb_0} => {f_msb_0}_msb,
            {f_msb_1} => {f_msb_1}_msb,
            sum => sum_tree_msb
        );

    --------------------------------------------------------------------------
    -- Árvore LSB original
    --------------------------------------------------------------------------
    u_tree_lsb : tree_lsb
        generic map(
            n => n/2 - h
        )
        port map(
{port_map_lsb}
            sum => sum_tree_lsb
        );

    --------------------------------------------------------------------------
    -- Carry entre LSB e MSB
    --------------------------------------------------------------------------
    csum <= ('0' & {f_msb_0}_msb(h-1 downto 0)) +
            ('0' & {f_msb_1}_msb(h-1 downto 0)) +
            ('0' & sum_tree_lsb(n/2-1 downto n/2-h));

    --------------------------------------------------------------------------
    -- CSLA
    --------------------------------------------------------------------------
    sum_c0 <= {f_msb_0}_msb(n/2+h-1 downto h) +
              {f_msb_1}_msb(n/2+h-1 downto h);

    sum_c1 <= ('0' & {f_msb_0}_msb(n/2+h-1 downto h)) +
              ('0' & {f_msb_1}_msb(n/2+h-1 downto h)) +
              to_unsigned(1, n/2+1);

    --------------------------------------------------------------------------
    -- MUX
    --------------------------------------------------------------------------
    sum_msb <= sum_c0 when csum(h) = '0' else sum_c1(n/2-1 downto 0);

    --------------------------------------------------------------------------
    -- Soma final truncada
    --------------------------------------------------------------------------
    sum <= sum_msb &
           csum(h-1 downto 0) &
           sum_tree_lsb(n/2-h-1 downto 0);

end rtl;
"""

        return vhdl

    
    def generate_tree_lsb_vhdl(self, m: int) -> str:
        if not self.is_power_of_two(m):
            raise ValueError("O número de vetores M deve ser potência de 2: 2, 4, 8, 16, 32...")

        if m < 2:
            raise ValueError("M deve ser maior ou igual a 2.")

        ports = [f"a{i}" for i in range(m)]
        h = int(math.log2(m))

        port_decl = self.format_port_decl(ports, indent="        ")

        vhdl = []

        vhdl.append(f"""library ieee;
use ieee.std_logic_1164.all;
use ieee.numeric_std.all;

entity tree_lsb is
    generic(n : integer := 16);
    port (
{port_decl} : in unsigned(n-1 downto 0);
        sum : out unsigned(n+{h-1} downto 0)
    );
end tree_lsb;

architecture rtl of tree_lsb is
    """)

        current_signals = ports.copy()
        levels = []
        level = 1

        while len(current_signals) > 1:
            next_count = len(current_signals) // 2
            next_signals = [f"flsb_{level}_{i}" for i in range(next_count)]

            levels.append((level, current_signals, next_signals))

            width_expr = "n" if level == 1 else f"n+{level-1}"

            vhdl.append(f"    -- nível {level}")
            for group in self.chunks(next_signals, 4):
                vhdl.append(
                    f"    signal {', '.join(group)} : unsigned({width_expr} downto 0);"
                )
            vhdl.append("")

            current_signals = next_signals
            level += 1

        vhdl.append("begin")
        vhdl.append("")

        for level, inputs, outputs in levels:
            vhdl.append("    --------------------------------------------------------------------------")
            vhdl.append(f"    -- Nível {level} ({len(outputs)} somas)")
            vhdl.append("    --------------------------------------------------------------------------")

            for i, out_sig in enumerate(outputs):
                in0 = inputs[2 * i]
                in1 = inputs[2 * i + 1]
                vhdl.append(f"    {out_sig} <= ('0' & {in0}) + ('0' & {in1});")

            vhdl.append("")

        final_signal = current_signals[0]

        vhdl.append("""    
    --------------------------------------------------------------------------
    -- Resultado final
    --------------------------------------------------------------------------""")
        vhdl.append(f"    sum <= {final_signal};")
        vhdl.append("")
        vhdl.append("end architecture;")

        return "\n".join(vhdl)

    def generate_tree_msb_vhdl(self, m: int) -> str:
        if not self.is_power_of_two(m):
            raise ValueError("O número de vetores M deve ser potência de 2: 2, 4, 8, 16, 32...")

        if m < 4:
            raise ValueError("A tree_msb precisa de pelo menos 4 vetores para expor dois ramos finais.")

        ports = [f"a{i}" for i in range(m)]
        h = int(math.log2(m))
        last_level = h - 1

        f_msb_0 = f"f{last_level}_0"
        f_msb_1 = f"f{last_level}_1"

        port_decl = self.format_port_decl(ports, indent="        ")

        vhdl = []

        vhdl.append(f"""library ieee;
use ieee.std_logic_1164.all;
use ieee.numeric_std.all;

entity tree_msb is
    generic(n : integer := 16);
    port (
{port_decl} : in unsigned(n-1 downto 0);
        {f_msb_1}, {f_msb_0} : out unsigned(n-1 downto 0);
        sum : out unsigned(n-1 downto 0)
    );
end tree_msb;

architecture rtl of tree_msb is
    """)

        current_signals = ports.copy()
        levels = []
        level = 1

        while len(current_signals) > 2:
            next_count = len(current_signals) // 2
            next_signals = [f"flsb_{level}_{i}" for i in range(next_count)]

            levels.append((level, current_signals, next_signals))

            vhdl.append(f"    -- nível {level}")
            for group in self.chunks(next_signals, 4):
                vhdl.append(
                    f"    signal {', '.join(group)} : unsigned(n-1 downto 0);"
                )
            vhdl.append("")

            current_signals = next_signals
            level += 1

        vhdl.append("begin")
        vhdl.append("")

        for level, inputs, outputs in levels:
            vhdl.append("    --------------------------------------------------------------------------")
            vhdl.append(f"    -- Nível {level} ({len(outputs)} somas)")
            vhdl.append("    --------------------------------------------------------------------------")

            for i, out_sig in enumerate(outputs):
                in0 = inputs[2 * i]
                in1 = inputs[2 * i + 1]
                vhdl.append(f"    {out_sig} <= {in0} + {in1};")

            vhdl.append("")

        vhdl.append("""    --------------------------------------------------------------------------
        -- Resultado final
        --------------------------------------------------------------------------""")
        vhdl.append(f"    sum <= {current_signals[0]} + {current_signals[1]};")
        vhdl.append("")
        vhdl.append(f"    {f_msb_0} <= {current_signals[0]};")
        vhdl.append(f"    {f_msb_1} <= {current_signals[1]};")
        vhdl.append("")
        vhdl.append("end architecture;")

        return "\n".join(vhdl)
        
    def generate_pins(self, m: int) -> str:
        if not self.is_power_of_two(m):
            raise ValueError("O número de vetores M deve ser potência de 2: 2, 4, 8, 16, 32...")

        entity_name = f"sum{m}xn"

        # Agora o TOP só tem duas entradas: a0 e a1
        ports = ["a0", "a1"]

        qsf = []

        qsf.append(
f"""
# Project-Wide Assignments
# ========================
set_global_assignment -name ORIGINAL_QUARTUS_VERSION 25.1STD.0
set_global_assignment -name PROJECT_CREATION_TIME_DATE "20:49:01  MAY 06, 2026"
set_global_assignment -name LAST_QUARTUS_VERSION "25.1std.0 Standard Edition"
set_global_assignment -name VHDL_FILE vhdl/{entity_name}.vhd
set_global_assignment -name VHDL_FILE vhdl/msb_tree.vhd
set_global_assignment -name VHDL_FILE vhdl/lsb_tree.vhd
set_global_assignment -name PROJECT_OUTPUT_DIRECTORY output_files

# Classic Timing Assignments
# ==========================
set_global_assignment -name MIN_CORE_JUNCTION_TEMP 0
set_global_assignment -name MAX_CORE_JUNCTION_TEMP 85

# Analysis & Synthesis Assignments
# ================================
set_global_assignment -name FAMILY "Stratix V"
set_global_assignment -name TOP_LEVEL_ENTITY {entity_name}

# Fitter Assignments
# ==================
set_global_assignment -name DEVICE 5SEEBH40C2
set_global_assignment -name ERROR_CHECK_FREQUENCY_DIVISOR 4

# EDA Netlist Writer Assignments
# ==============================
set_global_assignment -name EDA_SIMULATION_TOOL "Questa Altera FPGA (Verilog)"
set_global_assignment -name EDA_TIME_SCALE "1 ps" -section_id eda_simulation
set_global_assignment -name EDA_OUTPUT_DATA_FORMAT "VERILOG HDL" -section_id eda_simulation
"""
    )

        for p in ports:
            qsf.append(f"set_instance_assignment -name VIRTUAL_PIN ON -to {p}")

        qsf.append(
f"""
set_instance_assignment -name VIRTUAL_PIN ON -to sum

# start DESIGN_PARTITION(Top)
# ---------------------------
set_global_assignment -name PARTITION_NETLIST_TYPE SOURCE -section_id Top
set_global_assignment -name PARTITION_FITTER_PRESERVATION_LEVEL PLACEMENT_AND_ROUTING -section_id Top
set_global_assignment -name PARTITION_COLOR 16764057 -section_id Top
# end DESIGN_PARTITION(Top)
# -------------------------

# end ENTITY({entity_name})
# -------------------------
set_global_assignment -name POWER_PRESET_COOLING_SOLUTION "23 MM HEAT SINK WITH 200 LFPM AIRFLOW"
set_global_assignment -name POWER_BOARD_THERMAL_MODEL "NONE (CONSERVATIVE)"
set_global_assignment -name FITTER_EFFORT "STANDARD FIT"
set_instance_assignment -name PARTITION_HIERARCHY root_partition -to | -section_id Top
"""
    )

        return "\n".join(qsf)
    
    def create_files(self, path: str):
        output_dir = Path(path)
        output_dir.mkdir(parents=True, exist_ok=True)
        (output_dir / "vhdl").mkdir(parents=True, exist_ok=True)

        vhdl_content = self.generate_sum_vhdl(self.m)
        lsb_tree_content = self.generate_tree_lsb_vhdl(self.m)
        msb_tree_content = self.generate_tree_msb_vhdl(self.m)
        qsf_content = self.generate_pins(self.m)

        (output_dir / "vhdl" / f"{self.entity_name}.vhd").write_text(
            vhdl_content,
            encoding="utf-8"
        )

        (output_dir / "vhdl" / "lsb_tree.vhd").write_text(
            lsb_tree_content,
            encoding="utf-8"
        )

        (output_dir / "vhdl" / "msb_tree.vhd").write_text(
            msb_tree_content,
            encoding="utf-8"
        )

        (output_dir / "generic_propose_tree.qsf").write_text(
            qsf_content,
            encoding="utf-8"
        )


In [21]:
x = createFiles(32)
x.create_files(".")

In [22]:
base_dir = Path(r"C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_trunc")

for m in [16, 32, 64, 128, 256, 512]:
    target = base_dir / f"v{m}"
    createFiles(m).create_files(target)